In [208]:
import os
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from matplotlib.patches import Ellipse
from astropy.visualization import ZScaleInterval


In [209]:
FITS_PATH = "../data/raw/fits/jw02107-o039_t018_miri_f770w_i2d.fits"

CATALOGUE_PATH = "../data/processed/metadata/segregatedMergedCatalogue.txt"

SIZE_BOUNDS_PATH = "../data/processed/metadata/curriculum/sizeClassBounds.csv"

OUTPUT_PATH = "../outputs/figures"

CURRICULUM_STAGES = {
    "stage00": ["edgeLarge"],
    "stage01": ["giants", "edgeLarge"],
    "stage02": ["large", "giants", "edgeLarge"],
    "stage03": ["medium", "large", "giants", "edgeLarge"],
    "stage04": ["small", "medium", "large", "giants", "edgeLarge"],
    "stage05": ["tiny", "small", "medium", "large", "giants", "edgeLarge"],
}

ACTIVE_STAGE = "stage04"

os.makedirs(OUTPUT_PATH, exist_ok=True)


In [210]:
hdu = fits.open(FITS_PATH)
image = hdu[1].data
wcs = WCS(hdu[1].header)

print("Image shape:", image.shape)

# Conversion constants
distance_pc = 9.77e6

cdelt1 = abs(hdu[1].header["CDELT1"])
cdelt2 = abs(hdu[1].header["CDELT2"])

pixel_scale_arcsec = ((cdelt1 + cdelt2) / 2) * 3600
pc_per_arcsec = distance_pc / 206265
pc_per_pixel = pixel_scale_arcsec * pc_per_arcsec

print("pc_per_pixel:", pc_per_pixel)


Image shape: (1177, 2380)
pc_per_pixel: 5.254140159227344


Set DATE-AVG to '2022-07-17T12:01:53.586' from MJD-AVG.
Set DATE-END to '2022-07-17T12:54:46.016' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -37.754891 from OBSGEO-[XYZ].
Set OBSGEO-H to 1738895745.202 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


In [239]:
def save_stratified_overlay(df, selected_group, catalogue_output_path, active_stage, vmin, vmax):
    if len(df) == 0:
        print(f"WARNING: No bubbles found for selected_group = {selected_group}")
        return None

    avg_rad_pc_min = df["AVG_RAD_PC"].min()
    avg_rad_pc_max = df["AVG_RAD_PC"].max()

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(image, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

    for _, row in df.iterrows():
        bubble_id = int(row["GLOBAL_ID"])

        coord = SkyCoord(
            row["RA_DMS"],
            row["DEC_DMS"],
            frame="icrs"
        )
        
        x, y = wcs.world_to_pixel(coord)

        semi_major_pix = row["SEMI_MAJ_PC"] / pc_per_pixel
        semi_minor_pix = row["SEMI_MIN_PC"] / pc_per_pixel

        # PA to matplotlib convention
        # theta = -(row["PA_DEG"] - 90)
        theta = (row["PA_DEG"])

        ellipse = Ellipse(
            (x, y),
            width=2*semi_major_pix,
            height=2*semi_minor_pix,
            angle=theta,
            edgecolor='red',
            facecolor='none',
            linewidth=2
        )
        
        ax.add_patch(ellipse)

        ax.text(
            x,
            y,
            str(bubble_id),
            color="red",
            fontsize=5,
            ha="center",
            va="center"
        )

    ax.set_title(
        f"Segregation Stage {ACTIVE_STAGE}: {selected_group} bubbles, "
        f"AVG_RAD_PC {avg_rad_pc_min}–{avg_rad_pc_max} pc"
    )
    ax.set_xlabel("X pixel")
    ax.set_ylabel("Y pixel")

    plt.tight_layout()

    fig.savefig(catalogue_output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved overlay:", catalogue_output_path)
    return catalogue_output_path

In [240]:
df = pd.read_csv(CATALOGUE_PATH)



In [241]:
df.head()

,GLOBAL_ID,ID,RA_DMS,DEC_DMS,SEMI_MAJ_PC,SEMI_MIN_PC,AVG_RAD_PC,PA_DEG,ARM,DIST_ARM_PC,GAL_RAD_KPC,CATALOGUE
0,1,1,24d08m55.9s,15d48m7.9s,86,86,86,0,3,145,5.22,A
1,2,2,24d08m56.7s,15d48m2.9s,24,24,24,0,3,127,5.05,A
2,3,3,24d08m57.0s,15d48m9.0s,42,42,42,0,3,138,5.21,A
3,4,4,24d08m58.0s,15d47m44.9s,25,25,25,0,3,101,4.54,A
4,5,5,24d08m58.2s,15d48m2.6s,24,24,24,0,3,114,4.98,A


In [242]:
bounds = pd.read_csv(SIZE_BOUNDS_PATH, index_col="SIZE_CLASS")
lo, hi = bounds.loc[CURRICULUM_STAGES[ACTIVE_STAGE][0]]
df = df[df["AVG_RAD_PC"].between(lo, hi)]


In [243]:
map_output_path = os.path.join(OUTPUT_PATH, ACTIVE_STAGE+"SegregatedMergedCatalogueMap.png")

interval = ZScaleInterval()
vmin, vmax = interval.get_limits(image)

save_stratified_overlay(
        df=df,
        selected_group=CURRICULUM_STAGES[ACTIVE_STAGE],
        catalogue_output_path=map_output_path,
        active_stage=ACTIVE_STAGE,
        vmin=vmin, 
        vmax=vmax
    )

Saved overlay: ../outputs/figures/stage04SegregatedMergedCatalogueMap.png


'../outputs/figures/stage04SegregatedMergedCatalogueMap.png'